In [ ]:
import torch
def farthest_point_sample(H, npoint, xyz_surf, wid=64):
    """
    最远点采样
    随机选择一个初始点作为采样点，循环的将与当前采样点距离最远的点当作下一个采样点，直至满足采样点的数量需求
    """
    xyz = xyz_surf[:,:3,:].permute(0,2,1).unsqueeze(1)
    xyz = xyz.expand(-1,H,-1,-1)
    # xyz = xyz.permute(0,2,1,3)
    device = xyz.device
    B, _, N, C = xyz.shape
    npoint = min(npoint, N)
    centroids = torch.zeros(B, H, npoint, dtype=torch.long).to(device)
    distance = torch.ones(B, H, N).to(device) * 1e10  # 每个点与最近采样点的最小距离
    # farthest = torch.argmin(xyz.mean(-1), dim=-1).to(device)
    farthest = torch.randint(0, N, (B, H), dtype=torch.long).to(device)
    if wid is not None:
        group_indices = torch.zeros((B, H, npoint, wid), dtype=torch.long).to(device)
    else:
        group_indices = None

    for i in range(npoint):
        centroids[:, :, i] = farthest
        indices = farthest.unsqueeze(-1).unsqueeze(-1)
        indices = indices.expand(-1,-1,-1,C)
        centroid = torch.gather(xyz, dim=2, index=indices).contiguous().view(B, H, 1, -1) 
        dist = torch.nn.functional.pairwise_distance(xyz, centroid)
        mask = dist < distance
        distance[mask] = dist[mask]
        farthest = torch.max(distance, -1)[1]
        if wid is not None:
            group_indices[:,:,i,:] = torch.topk(dist, wid, dim=-1, largest=False)[1]
    return centroids.permute(0,2,1), group_indices.permute(0,2,1,3), xyz

In [ ]:
import trimesh
stl_file = 'E:\\MHATT_simp7_SS\\ShapeSource\\X-33.stl'
mesh_data = trimesh.load(stl_file)
points = torch.tensor(mesh_data.vertices)
faces = mesh_data.faces

In [ ]:
points

In [ ]:



xyzn_surf_source = torch.concat((points,points),dim=1)
xyzn_surf_source = xyzn_surf_source.unsqueeze(0).permute(0,2,1).float()
idx = farthest_point_sample(H=1, npoint=2048, xyz_surf=xyzn_surf_source, wid=round(8*xyzn_surf_source.shape[-1]/2048))[1]
idx = idx[0,:,0,0]
xyzn_surf_source = xyzn_surf_source[:,:,idx]
# xyzn_surf_source = torch.rand(1, 6, 2048)
surf_fps = farthest_point_sample(H=4, npoint=256*3, xyz_surf=xyzn_surf_source, wid=round(8*xyzn_surf_source.shape[-1]/2048))[1]# [B,FPS,H,wid]
surf_fps_size = [surf_fps.shape[0], 3, 256, surf_fps.shape[-2], surf_fps.shape[-1]]
surf_fps = surf_fps.reshape(surf_fps_size) # [B,N-att,fps,H,wid], surf_fps(new)[0,1,0,0,:]==surf_fps[0,512,0,:]
surf_fps = [surf_fps[:,i].unsqueeze(1) for i in range(3)]# list: [B,1(C),fps,H,wid]

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=xyzn_surf_source[0, 0, :],
    y=xyzn_surf_source[0, 1, :],
    z=xyzn_surf_source[0, 2, :],
    mode='markers',
    marker=dict(
        size=2,
        colorscale='viridis',
        opacity=1
    )
))
fig.update_layout(width=600, height=400)
fig.update_layout(scene={
    "camera": {
        "projection": {
            "type": "orthographic"
        }
    },
    "xaxis": {"showticklabels": False},
    "yaxis": {"showticklabels": False},
    "zaxis": {"showticklabels": False}
})
fig.show()

In [ ]:
import plotly.graph_objects as go
idx = surf_fps[0][0,0,:,0,0]
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=xyzn_surf_source[0, 0, idx],
    y=xyzn_surf_source[0, 1, idx],
    z=xyzn_surf_source[0, 2, idx],
    mode='markers',
    marker=dict(
        size=2,
        colorscale='viridis',
        opacity=1
    )
))
fig.update_layout(width=600, height=400)
fig.update_layout(scene={
    "camera": {
        "projection": {
            "type": "orthographic"
        }
    },
    "xaxis": {"showticklabels": False},
    "yaxis": {"showticklabels": False},
    "zaxis": {"showticklabels": False}
})
fig.show()

In [ ]:
idx = surf_fps[0][0,0,:,0,0].reshape(-1)
xyzn_surf_source[0,1,idx]

In [ ]:
import plotly.graph_objects as go

target = 4
fig = go.Figure()
idx = surf_fps[0][0,0,:,0,0].reshape(-1)
target = torch.argmin(xyzn_surf_source[0, 0, idx]).item()
idx = surf_fps[0][0,0,target,0,0].reshape(-1)
fig.add_trace(go.Scatter3d(
    x=xyzn_surf_source[0, 0, idx],
    y=xyzn_surf_source[0, 1, idx],
    z=xyzn_surf_source[0, 2, idx],
    mode='markers',
    marker=dict(
        size=2,
        color='blue',
        opacity=1
    ),
    name='FPS Centroid'
))
idx = surf_fps[0][0,0,target,0,1:].reshape(-1)
fig.add_trace(go.Scatter3d(
    x=xyzn_surf_source[0, 0, idx],
    y=xyzn_surf_source[0, 1, idx],
    z=xyzn_surf_source[0, 2, idx],
    mode='markers',
    marker=dict(
        size=2,
        color='red',
        opacity=1
    ),
    name='Group'
))
# idx = torch.concat((surf_fps[0][0,0,:target,0,1:].reshape(-1), surf_fps[0][0,0,target+1:,0,1:].reshape(-1)))
idx = surf_fps[0][0,0,:,0,:].reshape(-1)
fig.add_trace(go.Scatter3d(
    x=xyzn_surf_source[0, 0, idx],
    y=xyzn_surf_source[0, 1, idx],
    z=xyzn_surf_source[0, 2, idx],
    mode='markers',
    marker=dict(
        size=1,
        color='grey',
        opacity=1
    ),
    showlegend=False
))

fig.update_layout(width=450, height=400)
fig.update_layout(scene={
    "camera": {
        "projection": {
            "type": "orthographic"
        }
    },
    "xaxis": {"showticklabels": False},
    "yaxis": {"showticklabels": False},
    "zaxis": {"showticklabels": False}
})
# fig.update_layout(margin=dict(l=0, r=0.1, b=0, t=0))# tight layout b=0, t=10
fig.show()

In [ ]:

vol.shape

In [ ]:
import plotly.graph_objects as go
vol = torch.rand(10000,3)*2-1
idx = torch.sqrt((vol**2).sum(-1))<1
vol = vol[idx,:]
vol = vol[torch.randperm(vol.shape[0])[:2048]]
vol = vol.unsqueeze(0).permute(0,2,1)
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=vol[0, 0, :],
    y=vol[0, 1, :],
    z=vol[0, 2, :],
    mode='markers',
    marker=dict(
        size=1,
        color='grey',
        opacity=1
    )
))
fig.update_layout(width=600, height=400)
fig.update_layout(scene={
    "camera": {
        "projection": {
            "type": "orthographic"
        }
    },
    "xaxis": {"showticklabels": False},
    "yaxis": {"showticklabels": False},
    "zaxis": {"showticklabels": False}
})
fig.show()

In [ ]:
# 检查同一H同一FPS是不是特别近
# 不同FPS是不是特别远
# 不同H是不是无序分布
surf_fps[0].shape == torch.Size([1, 1, 256, 4, 8])


In [ ]:
# 检查同一H同一FPS是不是特别近
import plotly.graph_objects as go
label = -1*torch.ones((1,2,2048))
for i in range(surf_fps[0].shape[2]):#FPS
    for j in range(surf_fps[0].shape[3]): #H
        label[0,0,surf_fps[0][0,0,i,j,:]] = i
        label[0,1,surf_fps[0][0,0,i,j,:]] = j
    break
idx = 0
tmp = label[0,1,:] # pred_1d- gts_1d
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=xyzn_surf_source[0, 0, :],
    y=xyzn_surf_source[0, 1, :],
    z=xyzn_surf_source[0, 2, :],
    mode='markers',
    marker=dict(
        size=2,
        color=tmp,
        colorscale='viridis',
        opacity=1,
        colorbar=dict(title_text="Cp", len=200, lenmode="pixels", thickness=10, thicknessmode="pixels")
    )
))

In [ ]:
# 不同FPS是不是特别远
import plotly.graph_objects as go
label = -1*torch.ones((1,2,2048))
for i in range(surf_fps[0].shape[2]):#FPS
    for j in range(surf_fps[0].shape[3]): #H
        label[0,0,surf_fps[0][0,0,i,j,0]] = i
        label[0,1,surf_fps[0][0,0,i,j,0]] = j
        break
idx = 0
tmp = label[0,1,:] # pred_1d- gts_1d
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=xyzn_surf_source[0, 0, :],
    y=xyzn_surf_source[0, 1, :],
    z=xyzn_surf_source[0, 2, :],
    mode='markers',
    marker=dict(
        size=2,
        color=tmp,
        colorscale='viridis',
        opacity=1,
        colorbar=dict(title_text="Cp", len=200, lenmode="pixels", thickness=10, thicknessmode="pixels")
    )
))
fig.update_layout(width=600, height=400)
fig.show()
print(torch.sum(label>-1))

In [ ]:
# 不同H是不是无序分布
import plotly.graph_objects as go
label = -1*torch.ones((1,2,2048))
for i in range(surf_fps[0].shape[2]):#FPS
    for j in range(surf_fps[0].shape[3]): #H
        label[0,0,surf_fps[0][0,0,i,j,0]] = i
        label[0,1,surf_fps[0][0,0,i,j,0]] = j
        break
idx = 0
tmp = label[0,0,:] # pred_1d- gts_1d
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=xyzn_surf_source[0, 0, :],
    y=xyzn_surf_source[0, 1, :],
    z=xyzn_surf_source[0, 2, :],
    mode='markers',
    marker=dict(
        size=2,
        color=tmp,
        colorscale='viridis',
        opacity=1,
        colorbar=dict(title_text="Cp", len=200, lenmode="pixels", thickness=10, thicknessmode="pixels")
    )
))
fig.show()
print(torch.sum(label>-1))